# COMP 9130 — Capstone Project: Military Camouflage Object Detection
## Notebook 02: Training Experiment 3

**Author:** Binger Yu (Data & Preprocessing Lead)  
**Dataset:** COD10K + ACD1K + CAMO  
**Purpose:**
---
### Notebook Structure

In [ ]:
git add notebooks/02_train_exp3_Binger.ipynb
git commit -m "Update sweep cells for A100 (batch_size=16, accum_steps=1)"
git push

In [9]:
# Cell 0 — Pull the updated train_exp3.py
!git pull

Already up to date.


In [8]:
# Cell 1 — Clone repo
!git clone https://github.com/bing-er/AI-final-project.git
%cd /content/AI-final-project

fatal: destination path 'AI-final-project' already exists and is not an empty directory.
/content/AI-final-project


In [10]:
# Cell 2 — Install dependencies
!pip install -q transformers albumentations kaggle

In [11]:
# Cell 3 — Upload kaggle.json (run once per session)
import os
from google.colab import files

files.upload()  # select kaggle.json from your MacBook when prompted
os.makedirs('/root/.config/kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.config/kaggle/kaggle.json')
os.chmod('/root/.config/kaggle/kaggle.json', 0o600)
print('Kaggle credentials configured')

Saving kaggle.json to kaggle.json
Kaggle credentials configured


In [12]:
# Cell 4 — Download COD10K (~1.2 GB)
!kaggle datasets download \
    -d ivanomelchenkoim11/cod10k-dataset \
    -p data/ --unzip

Dataset URL: https://www.kaggle.com/datasets/ivanomelchenkoim11/cod10k-dataset
License(s): unknown
100% 2.26G/2.26G [00:16<00:00, 148MB/s]



In [13]:
# Cell 5 — Download ACD1K (~350 MB)
!kaggle datasets download \
    -d aalihhiader/military-camouflage-soldiers-dataset-mcs1k \
    -p data/ --unzip

Dataset URL: https://www.kaggle.com/datasets/aalihhiader/military-camouflage-soldiers-dataset-mcs1k
License(s): CC0-1.0
100% 372M/372M [00:02<00:00, 139MB/s]



In [14]:
# Upload CAMO zip from MacBook
from google.colab import files
uploaded = files.upload()  # select CAMO.zip when prompted

# Unzip to correct location
!unzip -q -n CAMO.zip -d data/ && echo "Done"

# Verify
import os
for p in ['data/CAMO-V.1.0-CVIU2019/Images/Train',
          'data/CAMO-V.1.0-CVIU2019/Images/Test',
          'data/CAMO-V.1.0-CVIU2019/GT']:
    print('✅' if os.path.exists(p) else '❌', p)

Saving CAMO.zip to CAMO (1).zip
Done
✅ data/CAMO-V.1.0-CVIU2019/Images/Train
✅ data/CAMO-V.1.0-CVIU2019/Images/Test
✅ data/CAMO-V.1.0-CVIU2019/GT


In [15]:
# Cell 7 — Verify folder structure
import os
expected = [
    'data/COD10K-v3/Train/Image',
    'data/COD10K-v3/Train/GT_Object',
    'data/COD10K-v3/Test/Image',
    'data/COD10K-v3/Test/GT_Object',
    'data/dataset-splitM/Training/images',
    'data/dataset-splitM/Training/GT',
    'data/dataset-splitM/Testing/images',
    'data/dataset-splitM/Testing/GT',
    'data/CAMO-V.1.0-CVIU2019/Images/Train',
    'data/CAMO-V.1.0-CVIU2019/Images/Test',
    'data/CAMO-V.1.0-CVIU2019/GT',
]
for p in expected:
    status = '✅' if os.path.exists(p) else '❌ NOT FOUND'
    print(f'{status} — {p}')

✅ — data/COD10K-v3/Train/Image
✅ — data/COD10K-v3/Train/GT_Object
✅ — data/COD10K-v3/Test/Image
✅ — data/COD10K-v3/Test/GT_Object
✅ — data/dataset-splitM/Training/images
✅ — data/dataset-splitM/Training/GT
✅ — data/dataset-splitM/Testing/images
✅ — data/dataset-splitM/Testing/GT
✅ — data/CAMO-V.1.0-CVIU2019/Images/Train
✅ — data/CAMO-V.1.0-CVIU2019/Images/Test
✅ — data/CAMO-V.1.0-CVIU2019/GT


In [17]:
# Cell 8 — Verify GPU
!nvidia-smi

Mon Mar 30 18:28:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [11]:
# Cell 9 — Verify dataset loading
!PYTHONPATH=/content/AI-final-project/src \
 python src/dataset.py data/ splits/

Verifying all conditions and splits...

--- COD10K / train ---
  [Splits] COD10K train: 5950 images (from cod10k_splits.json)
  [COD10K] 5950 images (split mode)
  [DataLoader] condition=cod10k split=train samples=5950 batches=1487
  image : torch.Size([4, 3, 512, 512])  mask : torch.Size([4, 1, 512, 512])  values : [0.0, 1.0]  datasets : {'COD10K'}
  ✅ OK

--- COD10K / val ---
  [Splits] COD10K val: 3950 images (from cod10k_splits.json)
  [COD10K] 3950 images (split mode)
  [DataLoader] condition=cod10k split=val samples=3950 batches=988
  image : torch.Size([4, 3, 512, 512])  mask : torch.Size([4, 1, 512, 512])  values : [0.0, 1.0]  datasets : {'COD10K'}
  ✅ OK

--- ACD1K / train ---
  [ACD1K] 748 images (scan mode)
  [DataLoader] condition=acd1k split=train samples=748 batches=187
  image : torch.Size([4, 3, 512, 512])  mask : torch.Size([4, 1, 512, 512])  values : [0.0, 1.0]  datasets : {'ACD1K'}
  ✅ OK

--- ACD1K / val ---
  [Splits] ACD1K val: 230 images (from acd1k_splits.json)


In [18]:
# Cell 10 — Sweep run 1 (lr=1e-4)
!PYTHONPATH=/content/AI-final-project/src \
 python src/train_exp3.py \
    --lr 1e-4 --acd1k_w 8.0 --epochs 20 \
    --batch_size 16 --accum_steps 1 \
    --data_root data/ --splits_dir splits/ \
    --output_dir outputs/exp3/sweep_lr1e4

Device: cuda
Config saved → outputs/exp3/sweep_lr1e4/config.json
{
  "data_root": "data/",
  "splits_dir": "splits/",
  "output_dir": "outputs/exp3/sweep_lr1e4",
  "lr": 0.0001,
  "acd1k_w": 8.0,
  "weight_decay": 0.0001,
  "epochs": 20,
  "batch_size": 8,
  "accum_steps": 2,
  "patience": 10,
  "num_workers": 2,
  "seed": 42
}

[DataLoaders]
  [Splits] COD10K train: 5950 images (from cod10k_splits.json)
  [COD10K] 5950 images (split mode)
  [CAMO] 1000 images (scan mode)
  [ACD1K] 748 images (scan mode)
  [Sampler] ACD1K weight=8.0, others=1.0
  [DataLoader] condition=joint split=train samples=7698 batches=962
  [Splits] ACD1K val: 230 images (from acd1k_splits.json)
  [ACD1K] 230 images (split mode)
  [DataLoader] condition=acd1k split=val samples=230 batches=29

[Model] Loading SegFormer-B2 (ADE20K pretrained)...
Loading weights: 100% 380/380 [00:00<00:00, 840.42it/s, Materializing param=segformer.encoder.patch_embeddings.3.proj.weight]
SegformerForSemanticSegmentation LOAD REPORT f

In [13]:
# Cell 11 — Sweep run 2 (lr=6e-5)
!PYTHONPATH=/content/AI-final-project/src \
 python src/train_exp3.py \
    --lr 6e-5 --acd1k_w 8.0 --epochs 20 \
    --batch_size 16 --accum_steps 1 \
    --data_root data/ --splits_dir splits/ \
    --output_dir outputs/exp3/sweep_lr6e5

Device: cuda
Config saved → outputs/exp3/sweep_lr6e5/config.json
{
  "data_root": "data/",
  "splits_dir": "splits/",
  "output_dir": "outputs/exp3/sweep_lr6e5",
  "lr": 6e-05,
  "acd1k_w": 8.0,
  "weight_decay": 0.0001,
  "epochs": 20,
  "batch_size": 8,
  "accum_steps": 2,
  "patience": 10,
  "num_workers": 2,
  "seed": 42
}

[DataLoaders]
  [Splits] COD10K train: 5950 images (from cod10k_splits.json)
  [COD10K] 5950 images (split mode)
  [CAMO] 1000 images (scan mode)
  [ACD1K] 748 images (scan mode)
  [Sampler] ACD1K weight=8.0, others=1.0
  [DataLoader] condition=joint split=train samples=7698 batches=962
  [Splits] ACD1K val: 230 images (from acd1k_splits.json)
  [ACD1K] 230 images (split mode)
  [DataLoader] condition=acd1k split=val samples=230 batches=29

[Model] Loading SegFormer-B2 (ADE20K pretrained)...
config.json: 6.88kB [00:00, 17.4MB/s]
pytorch_model.bin: 100% 110M/110M [00:01<00:00, 67.6MB/s]
Loading weights:  89% 338/380 [00:00<00:00, 796.11it/s, Materializing param=s

In [ ]:
# Cell 12 — Sweep run 3 (lr=1e-5)
!PYTHONPATH=/content/AI-final-project/src \
 python src/train_exp3.py \
    --lr 1e-5 --acd1k_w 8.0 --epochs 20 \
    --batch_size 16 --accum_steps 1 \
    --data_root data/ --splits_dir splits/ \
    --output_dir outputs/exp3/sweep_lr1e5

In [ ]:
# Cell 13 — Check sweep results, pick best LR
import json
for run in ['sweep_lr1e4', 'sweep_lr6e5', 'sweep_lr1e5']:
    path = f'outputs/exp3/{run}/history.json'
    if not os.path.exists(path):
        print(f'{run}: not found')
        continue
    with open(path) as f:
        h = json.load(f)
    best = max(h, key=lambda x: x['val_mIoU'])
    print(f"{run}: best val mIoU={best['val_mIoU']:.4f} @ epoch {best['epoch']}")

In [18]:
# Training + auto-save in one cell
!PYTHONPATH=/content/AI-final-project/src \
 python src/train_exp3.py \
    --lr 1e-4 --acd1k_w 8.0 --epochs 50 \
    --batch_size 8 --accum_steps 2 \
    --data_root data/ --splits_dir splits/ \
    --output_dir outputs/exp3/final

# Auto push history to GitHub immediately after training
!git config --global user.email "gyu42@my.bcit.ca"
!git config --global user.name "bing-er"
!git add outputs/exp3/final/history.json outputs/exp3/final/config.json
!git commit -m "Exp3 final training results"
!git push

# Download checkpoint
from google.colab import files
files.download('outputs/exp3/final/best_model.pth')
files.download('outputs/exp3/final/history.json')

Device: cuda
Config saved → outputs/exp3/final/config.json
{
  "data_root": "data/",
  "splits_dir": "splits/",
  "output_dir": "outputs/exp3/final",
  "lr": 0.0001,
  "acd1k_w": 8.0,
  "weight_decay": 0.0001,
  "epochs": 50,
  "batch_size": 8,
  "accum_steps": 2,
  "patience": 10,
  "num_workers": 2,
  "seed": 42
}

[DataLoaders]
  [Splits] COD10K train: 5950 images (from cod10k_splits.json)
  [COD10K] 5950 images (split mode)
  [CAMO] 1000 images (scan mode)
  [ACD1K] 748 images (scan mode)
  [Sampler] ACD1K weight=8.0, others=1.0
  [DataLoader] condition=joint split=train samples=7698 batches=962
  [Splits] ACD1K val: 230 images (from acd1k_splits.json)
  [ACD1K] 230 images (split mode)
  [DataLoader] condition=acd1k split=val samples=230 batches=29

[Model] Loading SegFormer-B2 (ADE20K pretrained)...
Loading weights: 100% 380/380 [00:00<00:00, 854.33it/s, Materializing param=segformer.encoder.patch_embeddings.3.proj.weight]
SegformerForSemanticSegmentation LOAD REPORT from: nvidia/

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>